In [ ]:
from semantic_text_splitter import MarkdownSplitter
from tqdm import tqdm
from pathlib import Path
import requests
import os
from dotenv import load_dotenv
import time
import numpy as np

In [ ]:
load_dotenv()
aiproxy_apikey = os.getenv("AIPROXY_TOKEN")

In [ ]:
def get_chunks(file_path, chunk_size=1000, overlap=200):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    
    # Use a slightly larger chunk size to allow overlap
    splitter = MarkdownSplitter(chunk_size + overlap)
    initial_chunks = splitter.chunks(content)

    # Add overlap by sliding window over each chunk
    final_chunks = []
    for chunk in initial_chunks:
        start = 0
        while start < len(chunk):
            end = start + chunk_size
            sub_chunk = chunk[start:end]
            if sub_chunk.strip():  # skip empty chunks
                final_chunks.append(sub_chunk)
            if end >= len(chunk):
                break
            start += chunk_size - overlap  # slide window with overlap

    return final_chunks

In [ ]:
def get_embedding(text: str) -> list:
    url = "https://aiproxy.sanand.workers.dev/openai/v1/embeddings"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {aiproxy_apikey}"
    }
    data = {
        "model": "text-embedding-3-small",
        "input": text
    }

    try:
        response = requests.post(url, headers=headers, json=data)
        response.raise_for_status()
        result = response.json()
        return result["data"][0]["embedding"]
    except Exception as e:
        print(f"Error fetching embedding: {e}")
        return []

In [ ]:
files = [*Path("markdown").glob("*.md"), *Path("markdown").rglob("*.md")]
all_chunks = []
all_embeddings = []
total_chunks = 0
file_chunks = {}
for file_path in files:
    chunks = get_chunks(file_path)
    file_chunks[file_path] = chunks
    total_chunks += len(chunks)

print(f"Total chunks to process: {total_chunks}")

In [ ]:
with tqdm(total=total_chunks, desc="Processing embeddings") as pbar:
    for file_path, chunks in file_chunks.items():
        for chunk in chunks:
            try:
                embedding = get_embedding(chunk)
                all_chunks.append(chunk)
                all_embeddings.append(embedding)
                pbar.set_postfix({"file": file_path.name, "chunks": len(all_chunks)})
            except Exception as e:
                print(f"Error processing chunk from {file_path}: {e}")
                continue
            finally:
                pbar.update(1)

In [ ]:
np.savez(
    "embeddings.npz",
    chunks=all_chunks,
    embeddings=all_embeddings
)